# E-FLOATCAL — replacing the dither assumption with a measurement

**Single notebook. Run all. ~1 hour on one GPU. Uses the `E_INV_P0_v3` root only, so it is
independent of Daxing and can run in a second session alongside D3/D4.**

## What this fixes

Section IV-H of the manuscript calibrates the detector by *modelling* the decoder's
pre-quantization float as

$$\tilde{F} = \text{stored integer} + U(-0.5,\,0.5)$$

then injecting, clipping and rounding once. That model is the reason the paper has to say
"under a uniform model of the pre-quantization fractional values", and it is the one
remaining entry in Limitations that is an assumption rather than a measurement:

> *A modelled, not measured, quantization path.* The detector calibration assumes uniform
> pre-quantization fractional values. The sampler's actual float output distribution was not
> captured, and the fingerprint is injected into already-generated images.

Generation is bit-deterministic, so nothing needs retraining. This notebook regenerates a
subset under the **recorded seeds**, captures the decoder's float tensor *before* it becomes
an 8-bit PNG, injects there, and re-fits the slope.

## Two results, both publishable

**G1 tests the assumption directly** — the distribution of the fractional part of the true
decoder float, against uniform, by Kolmogorov–Smirnov and by mean and variance. If it is
uniform, the model was right for a reason, not by luck.

**G3 compares the slopes.** The manuscript reports $1.4142\times10^{-2}$ per unit $\alpha$
under the dither model, with a lower 99 % limit of $1.3613\times10^{-2}$, and
$1.4345\times10^{-2}$ unquantized.

| outcome | what changes |
|---|---|
| slopes agree | §IV-H drops "under a uniform model"; the Limitations entry is deleted; the calibration becomes measured end to end |
| slopes differ | report the measured slope, recompute $\alpha_{\mathrm{equiv}}$ and the detection ratio, and say the dither model was optimistic or pessimistic by the measured factor |

Neither outcome is a failure. The second is the more interesting paper.

## Scope
This calibrates the **measurement path**, not the transfer bound. It cannot change
$\lambda_U$ or $\tau_U$ — those come from $\theta$ and $R_{\mathrm{real}}$, neither of which
involves injection. What it can change is the statement of how sensitive the detector is,
and therefore the ratio between the detectable amplitude and the bound.

In [ ]:
# %% 1 · mount, Hugging Face login, imports, configuration
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random, csv as _csv
from dataclasses import dataclass, asdict

HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive"
except Exception:
    DRIVE = os.environ.get("EINV_DRIVE", ".")

@dataclass
class CFG:
    STAGES: tuple = ("G0",)                 # G0 -> G1 -> G2 -> G3
    SRC:  str = "inv_channel/E_INV_P0_v3"   # the base study: fingerprints and adapters
    OUT:  str = "inv_channel/E_FLOATCAL"
    ARM:  str = "A_raw_s0_r16"              # adapter whose generations we recalibrate
    OWN:  str = "A"                         # inject and measure this device
    OTH:  str = "B"                         # paired comparator
    N_GEN: int = 500
    GEN_SEED_BASE: int = 770000             # MUST match the base study's bank for this arm
    CAPTION: str = "a photograph, sks style"
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    CROP: int = 1024
    ALPHAS: tuple = (0.0, 0.01, 0.02, 0.05, 0.10, 0.25)
    SD: str = "stabilityai/stable-diffusion-3.5-medium"
    # manuscript values under the dither model, for the comparison in G3
    REF_SLOPE_Q: float = 1.4142e-02
    REF_SLOPE_Q_LO99: float = 1.3613e-02
    REF_SLOPE_FLOAT: float = 1.4345e-02
    REF_INTERCEPT: float = -6.598e-05
C = CFG(); MEAS = C.CROP
SRC  = os.path.join(DRIVE, C.SRC)
ROOT = os.path.join(DRIVE, C.OUT)
for d in ("", "floats", "csv", "meta"): os.makedirs(os.path.join(ROOT, d), exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
def HALT(m): raise SystemExit(f"\n{'='*72}\n[HALT] {m}\n{'='*72}")

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],
               check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _p1; _p1.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as _p2; _p2.is_torchao_available = lambda: False
except Exception: pass
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DT  = torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN); print("[env] HF login ok")
else:
    print("[env] no HF token — G0 will fail on a gated model. Set HF_TOKEN.")

if not os.path.isdir(SRC): HALT(f"base study root not found: {SRC}")
ADIR = os.path.join(SRC, "adapters", C.ARM)
if not os.path.exists(os.path.join(ADIR, "pytorch_lora_weights.safetensors")):
    have = sorted(os.path.basename(p) for p in glob.glob(os.path.join(SRC,"adapters","*")))
    HALT(f"adapter {C.ARM} not found under {SRC}/adapters.\nAvailable: {have[:12]}")
FP = {}
for r, which in ((C.OWN,"E1"), (C.OWN,"E2"), (C.OTH,"E2")):
    p = os.path.join(SRC, "fingerprints", f"K_{r}_{which}.npy")
    if not os.path.exists(p): HALT(f"fingerprint missing: {p}")
    FP[(r,which)] = np.load(p)
print(f"[env] source {SRC}\n[env] arm {C.ARM} | inject K_{C.OWN}_E1 | measure "
      f"K_{C.OWN}_E2 vs K_{C.OTH}_E2 | {C.N_GEN} images | alphas {C.ALPHAS}")
if DEV == "cuda":
    print(f"[env] {torch.cuda.get_device_name(0)}  dtype {DT}")

def append_row(p,h,r):
    new = not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o, open(t,"w"), **kw); os.replace(t,p)
def fpath(i): return os.path.join(ROOT,"floats",f"{i:05d}.npy")

In [ ]:
# %% 2 · forensic core — identical to the base study
def _c2(x,k):
    xt = torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt = torch.from_numpy(k)[None,None]
    if DEV=="cuda": return tF.conv2d(xt.cuda(), kt.cuda(), padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0 = np.float32(4.0)
    co = pywt.wavedec2(img, "db8", level=4, mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch = ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v = np.maximum(_c2(ch*ch, np.ones((w,w),np.float32)/(w*w)) - s0, 0.0)
                vm = v if vm is None else np.minimum(vm, v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n = pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r = n - n.mean(); sd = float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def lum(rgb):
    return (0.299*rgb[...,0]+0.587*rgb[...,1]+0.114*rgb[...,2]).astype(np.float32)
def contrast(rgb):
    """Paired device contrast: own fingerprint minus the same-model comparator."""
    Y = lum(rgb); W = wres(Y)
    return ncc(W, Y*FP[(C.OWN,"E2")]) - ncc(W, Y*FP[(C.OTH,"E2")])
print("[core] ready")

In [ ]:
# %% 3 · G — generate, capture the true decoder float, measure. One pass, nothing stored.
#
# REPLACES cells G0, G1 and G2. Two problems with the previous version:
#
#   1. `np.save(path + ".tmp", arr)` silently appends ".npy" when the name does not already
#      end in it, so it wrote 00000.npy.tmp.npy and os.replace could not find the file it was
#      told to move. Same class as PIL inferring format from the extension. Where an atomic
#      temp write is wanted, pass a FILE HANDLE: np.save(fh, arr) does not rename anything.
#
#   2. Storing 500 float32 tensors at 1024x1024x3 is 5.9 GB written to Drive over a network
#      filesystem, for data used once and then discarded. Generation, the fractional-part
#      test and the injection sweep all happen in one pass here, so nothing is stored but the
#      CSV. Resumability comes from the CSV, which already keys on (idx, alpha).
#
# The decode runs through a SEPARATE float32 autoencoder rather than casting the bf16 pipeline
# VAE, which diffusers warns about; the whole point is an accurate pre-quantization float.
def g():
    from diffusers import StableDiffusion3Pipeline, AutoencoderKL
    if DEV != "cuda": HALT("G needs a GPU")
    p  = os.path.join(ROOT, "csv", "g2_measure.csv")
    fj = os.path.join(ROOT, "G1_fractional.json")
    done = {}
    if os.path.exists(p):
        for r in _csv.DictReader(open(p)):
            done.setdefault(int(r["idx"]), set()).add(r["alpha"])
    need = [i for i in range(C.N_GEN)
            if len(done.get(i, set())) < len(C.ALPHAS)]
    if not need:
        print(f"[G] all {C.N_GEN} images already measured at {len(C.ALPHAS)} amplitudes")
        return
    print(f"[G] {len(need)} images to generate and measure "
          f"({len(C.ALPHAS)} amplitudes each)\n")

    Kinj = FP[(C.OWN, "E1")]                       # independent of the measurement estimate
    pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True)
    pipe.load_lora_weights(ADIR)
    vae32 = AutoencoderKL.from_pretrained(C.SD, subfolder="vae",
                                          torch_dtype=torch.float32, token=True).to(DEV).eval()
    vsf = float(vae32.config.scaling_factor); vsh = float(vae32.config.shift_factor or 0.0)
    frac_samples, t0 = [], time.time()
    try:
        for s in range(0, len(need), C.GEN_BATCH):
            ix = need[s:s+C.GEN_BATCH]
            gs = [torch.Generator(device=DEV).manual_seed(C.GEN_SEED_BASE+i) for i in ix]
            with torch.inference_mode():
                lat = pipe(prompt=[C.CAPTION]*len(ix), num_inference_steps=C.GEN_STEPS,
                           guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS,
                           generator=gs, output_type="latent").images
                lat = (lat.to(torch.float32)/vsf) + vsh
                img = vae32.decode(lat, return_dict=False)[0]
            # the true pre-quantization float: [0,255], never rounded
            F = ((img/2 + 0.5).clamp(0, 1)*255.0).permute(0, 2, 3, 1).cpu().numpy().astype(np.float32)
            for k, i in enumerate(ix):
                Fi = F[k]
                if len(frac_samples) < 60:          # G1: fractional part of the true float
                    v = Fi.ravel()
                    frac_samples.append(v[np.random.default_rng(i).integers(0, v.size, 20000)])
                for a in C.ALPHAS:
                    if f"{a:.4f}" in done.get(i, set()): continue
                    G = Fi + a*Fi*Kinj[..., None]
                    Q = np.clip(G, 0, 255).round().astype(np.uint8).astype(np.float32)
                    append_row(p, ["idx","alpha","contrast"],
                               [i, f"{a:.4f}", f"{contrast(Q):.6e}"])
            if (s//C.GEN_BATCH) % 10 == 0:
                el = time.time()-t0
                print(f"    {s+len(ix)}/{len(need)}  eta "
                      f"{(el/max(s+len(ix),1))*(len(need)-s-len(ix))/60:.0f}min")
    finally:
        del pipe, vae32; gc.collect(); torch.cuda.empty_cache()

    # ---- G1: is the fractional part uniform? ----
    if frac_samples and not os.path.exists(fj):
        Fv   = np.concatenate(frac_samples)
        frac = Fv - np.floor(Fv)
        ks   = sps.kstest(frac, "uniform")
        res  = {"n":int(frac.size), "mean":float(frac.mean()), "var":float(frac.var()),
                "ks_stat":float(ks.statistic), "ks_p":float(ks.pvalue),
                "clipped_low":float((Fv<=0).mean()), "clipped_high":float((Fv>=255).mean()),
                "deciles":[float(x) for x in np.percentile(frac, np.arange(10,100,10))]}
        ajson(res, fj, indent=2)
        print(f"\n[G1] fractional part of the true decoder float, n={frac.size:,}")
        print(f"     mean {frac.mean():.4f} (uniform 0.5000) | var {frac.var():.5f} "
              f"(uniform {1/12:.5f})")
        print(f"     KS vs U(0,1): D={ks.statistic:.4f}, p={ks.pvalue:.3g}")
        print(f"     deciles {np.array2string(np.percentile(frac,np.arange(10,100,10)),precision=3)}")
        print(f"     saturated pixels: {100*(Fv<=0).mean():.3f}% at 0, "
              f"{100*(Fv>=255).mean():.3f}% at 255")
        print("     Read the MEAN, VARIANCE and DECILES, not the KS p-value — KS on millions")
        print("     of samples rejects any departure however small. The calibration depends on")
        print("     whether the model was MATERIALLY wrong.")
        if abs(frac.mean()-0.5) < 0.02 and abs(frac.var()-1/12) < 0.01:
            print("     => uniform to within 2% in mean and variance; the dither model was a")
            print("        good approximation. G3 confirms the slope.")
        else:
            print("     => NOT uniform. The dither model is misspecified and G3's measured")
            print("        slope is the one to report.")

    d = pd.read_csv(p); exp = C.N_GEN*len(C.ALPHAS)
    if len(d) != exp: HALT(f"G incomplete: {len(d)}/{exp} rows")
    print(f"\n[G] PASS — {len(d)} measurements, nothing written to disk but the CSV "
          f"({(time.time()-t0)/60:.0f} min)")
if stage("G0") or stage("G"): g()


In [ ]:
# %% 6 · G3 — fit the slope and compare against the dither model
def g3():
    d = pd.read_csv(os.path.join(ROOT,"csv","g2_measure.csv"))
    d["alpha"] = d["alpha"].astype(float); d["contrast"] = d["contrast"].astype(float)
    piv = d.pivot(index="idx", columns="alpha", values="contrast")
    al  = np.array(sorted(d.alpha.unique()))
    print(f"[G3] {piv.shape[0]} images x {len(al)} amplitudes, measured on the TRUE "
          f"decoder float\n")
    print(f"{'alpha':>7s}{'mean':>14s}{'SE':>12s}{'t':>9s}   vs alpha=0")
    base = piv[0.0].to_numpy()
    rows = []
    for a in al:
        v  = piv[a].to_numpy(); se = v.std(ddof=1)/np.sqrt(len(v))
        dv = v - base; dse = dv.std(ddof=1)/np.sqrt(len(dv))
        rows.append({"alpha":float(a), "mean":float(v.mean()), "se":float(se),
                     "t":float(v.mean()/se), "incr":float(dv.mean()),
                     "incr_se":float(dse), "incr_t":float(dv.mean()/dse) if dse>0 else np.nan})
        print(f"{a:7.3f}{v.mean():14.4e}{se:12.3e}{v.mean()/se:9.2f}   "
              + ("baseline" if a==0 else f"+{dv.mean():.3e} (t {dv.mean()/dse:+.1f})"))

    # paired slope: every image contributes its own regression through the amplitudes
    x = al - al.mean()
    sl = ((piv[al].to_numpy() - piv[al].to_numpy().mean(axis=1, keepdims=True)) * x).sum(1) \
         / (x**2).sum()
    slope = float(sl.mean()); sse = float(sl.std(ddof=1)/np.sqrt(len(sl)))
    lo99  = slope - sps.norm.ppf(0.99)*sse
    icept = float(piv[0.0].mean())
    ratio = slope/C.REF_SLOPE_Q
    out = {"slope_measured":slope, "slope_se":sse, "slope_lower99":float(lo99),
           "intercept":icept, "n_images":int(piv.shape[0]),
           "ref_slope_dither":C.REF_SLOPE_Q, "ref_slope_dither_lo99":C.REF_SLOPE_Q_LO99,
           "ref_slope_float":C.REF_SLOPE_FLOAT, "ref_intercept":C.REF_INTERCEPT,
           "ratio_measured_over_dither":float(ratio), "per_alpha":rows}
    ajson(out, os.path.join(ROOT,"G3_slope.json"), indent=2)

    print(f"\n{'':22s}{'slope':>14s}{'lower 99%':>14s}{'intercept':>14s}")
    print(f"{'MEASURED (this run)':22s}{slope:14.4e}{lo99:14.4e}{icept:14.4e}")
    print(f"{'dither model (paper)':22s}{C.REF_SLOPE_Q:14.4e}{C.REF_SLOPE_Q_LO99:14.4e}"
          f"{C.REF_INTERCEPT:14.4e}")
    print(f"{'unquantized (paper)':22s}{C.REF_SLOPE_FLOAT:14.4e}{'—':>14s}{'—':>14s}")
    print(f"\nmeasured / dither = {ratio:.4f}   "
          f"(slope SE {sse:.3e}, so {abs(slope-C.REF_SLOPE_Q)/sse:.1f} SE apart)")

    print("\n" + "="*72)
    if abs(ratio-1) < 0.05:
        print("SLOPES AGREE within 5%. The uniform-dither model was not merely convenient —")
        print("it reproduces the true decoder float to within measurement error.")
        print("ACTIONS: in Section IV-H drop 'under a uniform model of the pre-quantization")
        print("fractional values' and say the float was captured directly; delete the")
        print("Limitations entry 'A modelled, not measured, quantization path'; keep every")
        print("number as published and cite this run as the validation.")
    else:
        d_ = "OPTIMISTIC" if slope < C.REF_SLOPE_Q else "PESSIMISTIC"
        print(f"SLOPES DIFFER by {100*abs(ratio-1):.1f}%. The dither model was {d_}.")
        print("ACTIONS: report the MEASURED slope in Section IV-H and Table III; recompute")
        print("alpha_equiv (was 6.5e-03 nominal) and the detection-to-bound ratio (was 2.41x)")
        print("with it; state the dither model and its error as a methods finding.")
    print("\nNeither outcome touches lambda_U or tau_U — those come from theta and R_real,")
    print("and no injection enters either. What changes is the stated sensitivity of the")
    print("measurement path.")
    print("="*72)
if stage("G3"): g3()

## Running it

| stage | wall | disk | what to check |
|---|---|---|---|
| `("G0",)` | ~35 min | ~5.9 GB of float tensors | 500 captures, no HALT |
| `("G1",)` | ~1 min | — | fractional mean vs 0.5, variance vs 0.0833 |
| `("G2",)` | ~20 min | — | 3000 measurements |
| `("G3",)` | seconds | — | the slope comparison |

`C.GEN_SEED_BASE` **must match the seed bank the base study used for this arm**. If it
doesn't, you are calibrating on different images than the paper reports — still a valid
calibration, but no longer paired with the published numbers. Check
`E_INV_P0_v3/meta/` or the pilot notebook's config before running G0.

The float tensors are large (1024×1024×3 float32 ≈ 12 MB each). Delete
`E_FLOATCAL/floats/` once G2 has finished; G3 only needs the CSV.

### Reading G1
KS on several million samples will reject uniformity for any departure however small, so the
p-value is not the interesting output. The **mean, variance and deciles** are: they say
whether the model was *materially* wrong, which is what the calibration actually depends on.

### If you want it cheaper
`N_GEN = 200` cuts G0 to about 15 minutes and still gives a slope SE small enough to detect
a 5 % departure. The paper's calibration used 2500 images across five arms, so this is a
single-arm check either way.